# <font color="#418FDE" size="6.5" uppercase>**Feature Spaces**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Combine parallel transformations with FeatureUnion and make_union. 
- Process heterogeneous columns using ColumnTransformer and column selectors. 
- Inspect feature names, sparse thresholds, weights, and nested composite estimators. 


## **1. Parallel Feature Unions**

### **1.1. FeatureUnion Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_01_01.jpg?v=1783985982" width="250">



>* Run multiple transformations in parallel
>* Combine outputs into one feature matrix

>* Parallel branches preserve complementary feature views
>* Acts as one pipeline-ready transformer

>* Design branches to add distinct useful views
>* Watch feature size, cost, and model fit



In [ ]:
#@title Python Code - FeatureUnion Basics

# Demonstrate FeatureUnion with parallel feature branches.
# Each transformer sees the same input data.
# The union concatenates outputs into one matrix.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset for a quick example.
iris = load_iris()
X = iris.data

# Validate the expected two-dimensional feature table.
if X.ndim != 2 or X.shape[1] != 4:
    raise ValueError("Expected iris data with four numeric columns.")

# This branch keeps scaled versions of the original features.
scaled_branch = StandardScaler()

# This branch creates squared features from the same original input.
squared_branch = FunctionTransformer(
    lambda data: data ** 2,
    feature_names_out="one-to-one",
)

# FeatureUnion runs both branches in parallel, then joins columns.
feature_union = FeatureUnion(
    [("scaled", scaled_branch), ("squared", squared_branch)]
)

# Fit and transform once, just like any other transformer.
combined_features = feature_union.fit_transform(X)

# Feature names show which branch produced each output column.
feature_names = feature_union.get_feature_names_out(iris.feature_names)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original shape: {X.shape}")
print(f"Combined shape: {combined_features.shape}")
print(f"First five output names: {list(feature_names[:5])}")
print(f"First row, first five values: {np.round(combined_features[0, :5], 2)}")



### **1.2. make union shorthand**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_01_02.jpg?v=1783985984" width="250">



>* Build parallel features without manual branch names
>* Combine multiple transformations into one feature space

>* Speeds experimentation with multiple feature views
>* Shows parallel transforms joined before modeling

>* Shorthand suits simple, familiar parallel transformations
>* Explicit names improve tuning and maintainability



In [ ]:
#@title Python Code - make union shorthand

# Demonstrate make_union for parallel feature transformations.
# Each branch sees the same input data.
# The outputs join into one feature matrix.

import numpy as np
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_iris
from sklearn.pipeline import make_union
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset for a quick example.
iris = load_iris()
X = iris.data[:, :2]

# Validate the small input shape before transforming.
if X.shape != (150, 2):
    raise ValueError("Expected 150 rows and 2 selected features.")

# This branch keeps scaled versions of the original columns.
scaled_branch = StandardScaler()

# This branch creates squared versions of the same columns.
squared_branch = FunctionTransformer(
    lambda data: data ** 2,
    feature_names_out="one-to-one",
)

# make_union builds a FeatureUnion and names branches automatically.
feature_union = make_union(scaled_branch, squared_branch)

# Fit once, then concatenate both branch outputs side by side.
combined_features = feature_union.fit_transform(X)

# Inspect the automatic branch names and the wider feature space.
branch_names = list(feature_union.named_transformers.keys())
print("scikit-learn version:", sklearn_version)
print("Automatic branch names:", branch_names)
print("Original shape:", X.shape)
print("Combined shape:", combined_features.shape)

# Show one row so beginners can see the joined result.
first_row = np.round(combined_features[0], 2).tolist()
print("First combined row:", first_row)



### **1.3. Transformer Weights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_01_03.jpg?v=1783985986" width="250">



>* Control each transformer's influence in FeatureUnion
>* Balance feature groups before concatenation

>* Weight stronger representations more heavily
>* Scale outputs before final modeling

>* Validate weights to balance signal and noise
>* Adjust feature blocks without rebuilding unions



In [ ]:
#@title Python Code - Transformer Weights

# This example demonstrates weighted parallel feature transformations.
# FeatureUnion combines outputs from the same input data.
# Printed norms show how weights change feature influence.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset for a fast demonstration.
iris = load_iris()
X = iris.data

# Validate the expected two-dimensional feature matrix.
if X.ndim != 2 or X.shape[1] < 4:
    raise ValueError("Expected at least four numeric input columns.")

# These transformers create two different views of the same rows.
first_two = FunctionTransformer(lambda data: data[:, :2], validate=False)
last_two = FunctionTransformer(lambda data: data[:, 2:4], validate=False)

# Scaling keeps each block comparable before applying weights.
scaled_first_two = FeatureUnion([("select", first_two)])
scaled_last_two = FeatureUnion([("select", last_two)])

# Build one union without weights and one with transformer weights.
unweighted_union = FeatureUnion(
    [("sepal", first_two), ("petal", last_two)]
)

weighted_union = FeatureUnion(
    [("sepal", first_two), ("petal", last_two)],
    transformer_weights={"sepal": 0.5, "petal": 2.0},
)

# Standardize once so the weight effect is easy to see.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Transform the same scaled data through both unions.
X_unweighted = unweighted_union.fit_transform(X_scaled)
X_weighted = weighted_union.fit_transform(X_scaled)

# Compare average block sizes after concatenation.
sepal_unweighted = np.linalg.norm(X_unweighted[:, :2], axis=1).mean()
petal_unweighted = np.linalg.norm(X_unweighted[:, 2:], axis=1).mean()
sepal_weighted = np.linalg.norm(X_weighted[:, :2], axis=1).mean()
petal_weighted = np.linalg.norm(X_weighted[:, 2:], axis=1).mean()

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Combined feature shape: {X_weighted.shape}")
print(f"Unweighted mean norms: sepal={sepal_unweighted:.2f}, petal={petal_unweighted:.2f}")
print(f"Weighted mean norms: sepal={sepal_weighted:.2f}, petal={petal_weighted:.2f}")
print("Weights changed scale before side-by-side concatenation.")



## **2. Column Processing**

### **2.1. ColumnTransformer Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_02_01.jpg?v=1783985991" width="250">



>* Apply different preprocessing to different column types
>* Keep transformations organized in one estimator

>* Apply separate transformations to selected columns
>* Combine outputs into one reliable feature matrix

>* Use ColumnTransformer inside modeling pipelines
>* Keep preprocessing consistent, reliable, and maintainable



In [ ]:
#@title Python Code - ColumnTransformer Basics

# Demonstrate basic column-specific preprocessing.
# ColumnTransformer applies different steps by column.
# The output shows transformed feature names.

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
import sklearn

# Build a tiny heterogeneous dataset in memory.
customers = pd.DataFrame(
    {
        "age": [22, 35, 58, 41],
        "income_k": [45, 72, 110, 68],
        "region": ["North", "South", "North", "West"],
        "tier": ["Basic", "Gold", "Gold", "Basic"],
    }
)

# Select numeric and categorical columns explicitly.
numeric_columns = ["age", "income_k"]
categorical_columns = ["region", "tier"]

# Validate that every selected column exists.
selected_columns = numeric_columns + categorical_columns
missing_columns = set(selected_columns) - set(customers.columns)
if missing_columns:
    raise ValueError("A selected column is missing from the data.")

# Create one transformer for each column group.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(sparse_output=False), categorical_columns),
    ]
)

# Fit each transformer on its assigned columns, then concatenate outputs.
transformed_array = preprocessor.fit_transform(customers)
feature_names = preprocessor.get_feature_names_out()

# Show a compact transformed table for beginner inspection.
transformed_table = pd.DataFrame(
    transformed_array,
    columns=feature_names,
)

print("scikit-learn version:", sklearn.__version__)
print("Original shape:", customers.shape)
print("Transformed shape:", transformed_table.shape)
print("Feature names:", list(feature_names))
print("First transformed row:", transformed_table.iloc[0].round(2).tolist())



### **2.2. Column Selectors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_02_02.jpg?v=1783985987" width="250">



>* Select columns for appropriate preprocessing
>* Build reliable pipelines for heterogeneous data

>* Select columns by name, position, or type
>* Match selectors to each column’s modeling role

>* Choose selectors robust to dataset changes
>* Prevent wrong transformations in modeling pipelines



In [ ]:
#@title Python Code - Column Selectors

# This example routes columns by their data type.
# ColumnTransformer applies different preprocessing to each group.
# The output names reveal the created feature space.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Build a tiny mixed-type table in memory.
customers = pd.DataFrame(
    {
        "age": [25, 41, 36, 52],
        "income_k": [48, 82, 67, 110],
        "region": ["North", "South", "North", "West"],
        "tier": ["Silver", "Gold", "Gold", "Platinum"],
    }
)

# Select numeric and categorical columns automatically.
numeric_selector = make_column_selector(dtype_include="number")
categorical_selector = make_column_selector(dtype_include="object")

# Send each selected group to a suitable transformer.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_selector),
        ("cat", OneHotEncoder(sparse_output=False), categorical_selector),
    ]
)

# Fit once, then transform the same table for inspection.
transformed = preprocessor.fit_transform(customers)
feature_names = preprocessor.get_feature_names_out()

# Validate the transformed feature matrix shape.
if transformed.shape[1] != len(feature_names):
    raise ValueError("Feature names must match transformed columns.")

print("scikit-learn version:", sklearn.__version__)
print("Original columns:", list(customers.columns))
print("Numeric selector chose:", numeric_selector(customers))
print("Categorical selector chose:", categorical_selector(customers))
print("Transformed shape:", transformed.shape)
print("Feature names:", list(feature_names))



### **2.3. Remainder Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_02_03.jpg?v=1783985989" width="250">



>* Choose how unassigned columns are handled
>* Drop them or pass them through

>* Drop columns for tighter model control
>* Pass through only model-ready features

>* Preserve expected columns for reproducible preprocessing
>* Clarify feature origins and modeling decisions



In [ ]:
#@title Python Code - Remainder Handling

# Demonstrate ColumnTransformer remainder choices.
# Compare dropping and passing columns.
# Inspect resulting feature names clearly.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Build a tiny mixed-type customer table.
customers = pd.DataFrame(
    {
        "age": [22, 35, 58, 41],
        "income_k": [45, 72, 110, 68],
        "region": ["North", "South", "North", "West"],
        "loyalty_score": [0.20, 0.80, 0.60, 0.40],
        "notes": ["new", "vip", "vip", "new"],
    }
)

# Validate the columns used in this lesson.
needed_columns = {"age", "income_k", "region", "loyalty_score", "notes"}
if set(customers.columns) != needed_columns:
    raise ValueError("The teaching table has unexpected columns.")

# Transform selected columns and drop all others.
drop_remainder = ColumnTransformer(
    transformers=[
        ("scale_numbers", StandardScaler(), ["age", "income_k"]),
        ("encode_region", OneHotEncoder(sparse_output=False), ["region"]),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Transform selected columns and pass remaining columns through.
pass_remainder = ColumnTransformer(
    transformers=[
        ("scale_numbers", StandardScaler(), ["age", "income_k"]),
        ("encode_region", OneHotEncoder(sparse_output=False), ["region"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

# Fit both preprocessors on the same input table.
dropped_output = drop_remainder.fit_transform(customers)
passed_output = pass_remainder.fit_transform(customers)

# Collect feature names to show the remainder effect.
dropped_names = list(drop_remainder.get_feature_names_out())
passed_names = list(pass_remainder.get_feature_names_out())

print("scikit-learn version:", sklearn.__version__)
print("Original columns:", list(customers.columns))
print("With remainder='drop':", dropped_names)
print("Dropped output shape:", dropped_output.shape)
print("With remainder='passthrough':", passed_names)
print("Passed output shape:", passed_output.shape)
print("Remainder columns kept:", passed_names[-2:])



## **3. Composite Details**

### **3.1. Sparse Output Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_03_01.jpg?v=1783985978" width="250">



>* Mixed transformations create dense and sparse features
>* Thresholds guide memory-efficient output formats

>* Keep sparse when zeros dominate.
>* Use dense when storage overhead outweighs savings.

>* Nested branches can hide final sparsity
>* Threshold checks prevent memory and compatibility issues



In [ ]:
#@title Python Code - Sparse Output Thresholds

# This example inspects sparse output thresholds.
# Mixed feature blocks can change matrix format.
# Printed densities reveal the threshold decision.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# A tiny table mixes dense numeric and sparse categorical features.
data = pd.DataFrame(
    {
        "age": [22, 35, 58, 41, 29, 63],
        "spend": [40, 80, 20, 60, 55, 25],
        "city": ["A", "B", "C", "A", "D", "E"],
    }
)

# This validation keeps the example assumptions explicit.
if data.shape != (6, 3):
    raise ValueError("Expected six rows and three columns.")

# The same transformations are tested with two thresholds.
transformers = [
    ("numeric", StandardScaler(), ["age", "spend"]),
    ("city", OneHotEncoder(sparse_output=True), ["city"]),
]

# A low threshold converts the combined result to dense.
dense_preprocessor = ColumnTransformer(
    transformers=transformers,
    sparse_threshold=0.0,
)

dense_result = dense_preprocessor.fit_transform(data)
dense_is_sparse = hasattr(dense_result, "toarray")

# A high threshold preserves sparsity when zeros dominate.
sparse_preprocessor = ColumnTransformer(
    transformers=transformers,
    sparse_threshold=1.0,
)

sparse_result = sparse_preprocessor.fit_transform(data)
sparse_is_sparse = hasattr(sparse_result, "toarray")

# Density is the fraction of stored nonzero values.
nonzero_count = sparse_result.nnz
cell_count = sparse_result.shape[0] * sparse_result.shape[1]
density = nonzero_count / cell_count

feature_names = sparse_preprocessor.get_feature_names_out()
short_names = list(feature_names[:5])

print("scikit-learn composite sparse threshold demo")
print(f"Output shape: {sparse_result.shape[0]} rows x {sparse_result.shape[1]} features")
print(f"Overall density: {density:.2f}")
print(f"sparse_threshold=0.0 gives sparse output: {dense_is_sparse}")
print(f"sparse_threshold=1.0 gives sparse output: {sparse_is_sparse}")
print(f"First feature names: {short_names}")



### **3.2. Feature Name Inspection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_03_02.jpg?v=1783985977" width="250">



>* Trace transformed columns to original meanings
>* Make model signals interpretable and debuggable

>* Branches create different feature name patterns
>* Names verify transformations and final feature design

>* Diagnose model issues through feature names
>* Trace nested features for auditing and collaboration



In [ ]:
#@title Python Code - Feature Name Inspection

# Inspect names from a composite feature space.
# ColumnTransformer labels each transformed output feature.
# The printed names reveal feature origins.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Build a tiny mixed-type table in memory.
customers = pd.DataFrame(
    {
        "age": [25, 40, None, 35],
        "monthly_spend": [70.0, 120.0, 65.0, 90.0],
        "region": ["North", "South", "North", "West"],
        "plan": ["Basic", "Premium", "Basic", "Premium"],
    }
)

# Numeric and categorical branches create different feature names.
numeric_pipeline = make_pipeline(SimpleImputer(), StandardScaler())
categorical_pipeline = OneHotEncoder(sparse_output=False)

# Transformer names become prefixes in the final feature names.
preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, ["age", "monthly_spend"]),
        ("cat", categorical_pipeline, ["region", "plan"]),
    ],
    verbose_feature_names_out=True,
)

# Fit once, then inspect the transformed matrix and its labels.
transformed = preprocessor.fit_transform(customers)
feature_names = preprocessor.get_feature_names_out()

if transformed.shape[1] != len(feature_names):
    raise ValueError("Feature count and feature names do not match.")

print("scikit-learn version:", sklearn.__version__)
print("Original columns:", list(customers.columns))
print("Transformed shape:", transformed.shape)
print("First six feature names:", list(feature_names[:6]))
print("Last two feature names:", list(feature_names[-2:]))
print("Each name shows the branch and the derived feature.")



### **3.3. Nested Composite Estimators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_06/Lecture_B/image_03_03.jpg?v=1783985980" width="250">



>* Build workflows from nested reusable estimators
>* Organize varied preprocessing into one feature space

>* Named steps stay easy to manage
>* Teams can inspect transformed feature branches

>* Evaluate full workflows without data leakage
>* Inspect nested features for debugging and deployment



In [ ]:
#@title Python Code - Nested Composite Estimators

# Build a nested preprocessing and modeling workflow.
# Inspect names, sparsity, weights, and nested steps.
# See how raw columns become model features.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Create a tiny mixed-type customer dataset.
customer_data = pd.DataFrame(
    {
        "age": [25, 41, np.nan, 35, 52, 23, 46, 31],
        "income": [50, 82, 61, np.nan, 120, 45, 95, 70],
        "region": ["west", "east", "west", "south", "east", "south", "west", "east"],
        "comment": ["likes discounts", "premium service", "asks discounts", "new member", "premium member", "likes coupons", "service issue", "premium discounts"],
    }
)

# Store a small classification target for the final model.
target = np.array([0, 1, 0, 0, 1, 0, 1, 1])

# Validate the basic shape before fitting the workflow.
if len(customer_data) != len(target):
    raise ValueError("Rows and target labels must match.")

# Build reusable pipelines inside a ColumnTransformer.
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer()), ("scaler", StandardScaler())]
)

categorical_pipeline = Pipeline(
    [("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, ["age", "income"]),
        ("cat", categorical_pipeline, ["region"]),
        ("txt", CountVectorizer(), "comment"),
    ],
    transformer_weights={"num": 1.0, "cat": 2.0, "txt": 0.5},
    sparse_threshold=0.3,
)

# Nest the ColumnTransformer inside a full modeling pipeline.
model = Pipeline(
    [("prep", preprocessor), ("clf", LogisticRegression(random_state=42))]
)

model.fit(customer_data, target)

# Inspect the nested transformer and its produced feature space.
fitted_prep = model.named_steps["prep"]
feature_names = fitted_prep.get_feature_names_out()
transformed = fitted_prep.transform(customer_data)

# Print concise details that reveal the nested structure.
print("scikit-learn version:", sklearn.__version__)
print("Nested numeric steps:", list(numeric_pipeline.named_steps.keys()))
print("Transformer weights:", fitted_prep.transformer_weights)
print("Sparse threshold:", fitted_prep.sparse_threshold)
print("Output shape:", transformed.shape)
print("Sparse output:", hasattr(transformed, "toarray"))
print("First 8 feature names:", list(feature_names[:8]))



# <font color="#418FDE" size="6.5" uppercase>**Feature Spaces**</font>


In this lecture, you learned to:
- Combine parallel transformations with FeatureUnion and make_union. 
- Process heterogeneous columns using ColumnTransformer and column selectors. 
- Inspect feature names, sparse thresholds, weights, and nested composite estimators. 

In the next Module (Module 7), we will go over 'Feature Extraction'